# 08 Validate: do the tests pass?

The tests, one line each: the requirements (T), that the work is correct (V), and the
quality criteria (QC). Every test prints Pass or Fail, and at the end the whole table.

Saves `output/validation.csv`.

In [1]:
import csv
import time

import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import wafer

raw = np.load("../data/Wafer_Map_Datasets.npz")["arr_0"] # the original file, before cleaning
split = np.load("output/split.npz")
p = np.load("output/test_predictions.npz")
gen = np.load("output/generated.npz")
forest = joblib.load("output/forest.joblib")

parts = [split["maps_train"], split["maps_val"], split["maps_test"]]
part_labels = [split["y_train"], split["y_val"], split["y_test"]]
y_test, y_pred, y_base = p["y_test"], p["y_pred"], p["y_base"]

results = []


def check(number, what, passed, result):
    verdict = "Pass" if passed else "Fail" # short if/else on one line
    results.append([number, what, result, verdict])
    print(f"{number} {verdict}: {what} ({result})")

## 1. The requirements

In [2]:
# <= on sets means "only has stuff from", so only 0 and 1
check("T1", "8 answers per wafer, only 0 and 1",
      y_pred.shape[1] == 8 and set(np.unique(y_pred)) <= {0, 1}, f"shape {y_pred.shape}")

check("T2", "all zeros is called Normal", wafer.label_name(np.zeros(8)) == "Normal", wafer.label_name(np.zeros(8)))

# every map as 1 long row, unique keeps each different one once. same count = no copies
all_maps = np.concatenate(parts)
different = len(np.unique(all_maps.reshape(len(all_maps), -1), axis=0))
check("T3", "no copies left", different == len(all_maps), f"{len(all_maps):,} maps, {different:,} different")

check("T4", "no value 3 left", set(np.unique(all_maps)) <= {0, 1, 2}, f"values {np.unique(all_maps)}")

types_per_part = [len(np.unique(y, axis=0)) for y in part_labels]
check("T5", "all 38 types in every part", types_per_part == [38, 38, 38], f"types per part {types_per_part}")

T1 Pass: 8 answers per wafer, only 0 and 1 (shape (5534, 8))
T2 Pass: all zeros is called Normal (Normal)


T3 Pass: no copies left (36,890 maps, 36,890 different)


T4 Pass: no value 3 left (values [0 1 2])
T5 Pass: all 38 types in every part (types per part [38, 38, 38])


## 2. Is the work correct?

In [3]:
# V1: put two parts in one split set; if a map is in both, the split set has fewer different maps than rows
in_both = []
for a, b in [(0, 1), (0, 2), (1, 2)]: # train-val, train-test, val-test
    pile = np.concatenate([parts[a], parts[b]]).reshape(-1, 52 * 52) # -1 = figure out the number of rows yourself
    in_both.append(len(pile) - len(np.unique(pile, axis=0)))
check("V1", "no map in two split sets", sum(in_both) == 0, f"in common: {in_both}")

V1 Pass: no map in two split sets (in common: [0, 0, 0])


## 3. The quality criteria

In [4]:
recall = recall_score(y_test, y_pred, average=None)
precision = precision_score(y_test, y_pred, average=None)
f1 = f1_score(y_test, y_pred, average=None)
f1_base = f1_score(y_test, y_base, average=None)
all_right = accuracy_score(y_test, y_pred)

# argmin = where the lowest one is, so i can print the name
check("QC-01", "recall at least 70% on every defect", recall.min() >= 0.70,
      f"lowest {wafer.DEFECTS[recall.argmin()]} {recall.min() * 100:.1f}%")
check("QC-02", "all 8 right on at least 60%", all_right >= 0.60, f"{all_right * 100:.1f}%")
check("QC-03", "precision at least 60% on every defect", precision.min() >= 0.60,
      f"lowest {wafer.DEFECTS[precision.argmin()]} {precision.min() * 100:.1f}%")
check("QC-04", "higher F1 than the baseline on every defect", (f1 > f1_base).all(),
      f"higher on {(f1 > f1_base).sum()} of 8") # sum of Trues = how many it wins

recall_mine = recall_score(gen["labels"], gen["predicted"], average=None)
gap = ((recall - recall_mine) * 100).max()
check("QC-05", "my wafers at most 15 points lower", gap <= 15, f"biggest gap {gap:.1f} points")

# same timing as in 05
forest.set_params(n_jobs=1)
start = time.perf_counter()
for m in parts[2][:50]: # parts[2] = test
    forest.predict([wafer.features(m)])
ms = (time.perf_counter() - start) / 50 * 1000
check("QC-08", "one wafer under 50 ms", ms < 50, f"{ms:.1f} ms")

QC-01 Pass: recall at least 70% on every defect (lowest Random 72.8%)
QC-02 Pass: all 8 right on at least 60% (66.1%)
QC-03 Pass: precision at least 60% on every defect (lowest Scratch 75.9%)
QC-04 Fail: higher F1 than the baseline on every defect (higher on 7 of 8)
QC-05 Fail: my wafers at most 15 points lower (biggest gap 48.2 points)


QC-08 Pass: one wafer under 50 ms (9.0 ms)


## 4. The table

In [5]:
# markdown table so i can paste it in my R&V
print("| No | Result | Pass/Fail |")
print("|---|---|---|")
for number, what, result, verdict in results:
    print(f"| {number} | {result} | {verdict} |")

# newline="" or windows puts empty lines between the rows
with open("output/validation.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["No", "What", "Result", "Pass/Fail"])
    writer.writerows(results)
print("Saved output/validation.csv")

| No | Result | Pass/Fail |
|---|---|---|
| T1 | shape (5534, 8) | Pass |
| T2 | Normal | Pass |
| T3 | 36,890 maps, 36,890 different | Pass |
| T4 | values [0 1 2] | Pass |
| T5 | types per part [38, 38, 38] | Pass |
| V1 | in common: [0, 0, 0] | Pass |
| QC-01 | lowest Random 72.8% | Pass |
| QC-02 | 66.1% | Pass |
| QC-03 | lowest Scratch 75.9% | Pass |
| QC-04 | higher on 7 of 8 | Fail |
| QC-05 | biggest gap 48.2 points | Fail |
| QC-08 | 9.0 ms | Pass |
Saved output/validation.csv
